# TetraFT — heal_kl_50m (2-session resume)

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (**refresh** after code changes)
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`
- **Session B only:** Dataset with Session A `checkpoint-final` (**full** = opt+sched)

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run |
| Logical run | **one 50M KL job** (cosine over 12207 steps) |

**Recipe** (`heal_kl_50m`): skip GDN, λw=256, cosine→0.1, α=0.5 KL, T=2, β=0.01  
**CE bars:** scout_kl_5m ~49.3 @ 5M; heal_50m CE ~43.77 @ 50M; orig ~17.7

| Session | Steps | Tokens | Flags |
|---------|------:|-------:|-------|
| **A** | 0→6104 | ~25M | `save_optimizer=True`, fresh |
| **B** | 6104→12207 | ~25M | `--resume` full ckpt, `--skip-shock --skip-orig` |

**Session A go/no-go:** mid PPL ≲ ~50–52 and falling → upload ckpt + run B.  
**Disk:** one full final ckpt only (several GB). Clear `/kaggle/working` junk before zip.

Logic in `run_smoke.py` — notebook is glue only.


In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

from config import SMOKE_PRESETS
assert "heal_kl_50m" in SMOKE_PRESETS, "heal_kl_50m missing — refresh tetraft-code"
assert "scout_kl_5m" in SMOKE_PRESETS
print("presets ok:", sorted(SMOKE_PRESETS))

In [ ]:
from run_smoke import run_smoke
import argparse
import shutil
from pathlib import Path

# =============================================================================
# SESSION: "A" = first 25M (fresh, full ckpt) | "B" = resume → 50M
# =============================================================================
SESSION = "A"  # <-- set to "B" for second Kaggle run

PRESET = "heal_kl_50m"  # logical 50M; cosine horizon 12207 always
CLEAR_OUTPUT = True

if SESSION.upper() == "A":
    # Stop at ~25M; LR schedule still over full 50M (schedule_max_steps=12207)
    MAX_STEPS = 6104
    SAVE_OPTIMIZER = True  # REQUIRED for seamless Session B
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_A"
elif SESSION.upper() == "B":
    MAX_STEPS = 12207
    SAVE_OPTIMIZER = False  # final can be weights-only unless you need Session C
    RESUME = str(find_file("checkpoint-final"))  # from attached Session A Dataset
    SKIP_SHOCK = True
    SKIP_ORIG = True
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_B"
    print("resume from:", RESUME)
else:
    raise ValueError("SESSION must be 'A' or 'B'")

out = Path(OUTPUT_DIR)
if CLEAR_OUTPUT and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=MAX_STEPS,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=SKIP_SHOCK,
    skip_orig=SKIP_ORIG,
    resume=RESUME,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    schedule_max_steps=None,  # keep preset 12207
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    save_optimizer=SAVE_OPTIMIZER,
    skip_linear_attn=None,
    no_skip_linear_attn=False,
    distill_alpha=None,
    distill_temperature=None,
    quant_reg_beta=None,
    seed=42,
    device_map="auto",
)
print(
    f"SESSION={SESSION} preset={PRESET} max_steps={MAX_STEPS} "
    f"save_optimizer={SAVE_OPTIMIZER} resume={RESUME} out={OUTPUT_DIR}"
)
print("note: KL loads frozen FP teacher (~2× VRAM); full ckpt is large")
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
    "resumed_step", "schedule_horizon_steps", "distill",
]
print({k: results[k] for k in keys if k in results})
if "inventory_summary" in results:
    inv = results["inventory_summary"]
    print("inventory", inv)
    if inv.get("n_eligible", 0) > 150:
        print("WARNING: eligible looks like all-Linear — GDN skip may be off")
ppl = results.get("ppl_after_smoke")
if ppl is not None:
    ref = results.get("ppl_original") or results.get("ppl_original_ref") or 17.67
    print(f"after/orig ≈ {ppl / ref:.3f} (ref orig {ref})")
    if SESSION.upper() == "A":
        print(f"Session A mid PPL={ppl:.2f} — go/no-go: continue B if ≲50–52 and falling")
        print("Upload OUTPUT_DIR checkpoint-final (FULL) as Kaggle Dataset for Session B")
        print("Gate vs CE heal_25m ~48.2; scout was ~49.3 @ 5M")
    else:
        print(f"Session B final PPL={ppl:.2f} — bar: CE heal_50m ~43.77")
        print("PASS if < 43.77; strong if ≲ 35")

### Artifacts

**Session A** (`checkpoints_heal_kl_50m_A`):
- `checkpoint-final` — **must be full** (`weights_only: false`, has optimizer)
- `metrics.jsonl`, `smoke_results.json`, `linear_inventory.json`
- Optional `checkpoint-best` (weights-only OK)

**Session B:** attach A Dataset, set `SESSION = "B"`, refresh code if needed.

### Frozen baselines

| Run | ≈ tokens | Val PPL |
|-----|---------:|--------:|
| Original FP | — | ~17.7 |
| CE skip-GDN scout | 5.2M | ~60.6 |
| **scout_kl_5m** | **5.2M** | **~49.3** |
| CE heal_25m / heal_50m | 25M / 50M | ~48.2 / **~43.77** |
| heal_kl_50m A+B | 50M | **TBD** |

Record both session PPLs in `RESULTS.md`.
